# Datamine INPDDF Process Example

This notebook demonstrates how to configure and run the **`inpddf`** process wrapper in `dmstudio`.

## Process Description

## INPDDF

Input a complete file, including its Data Definition, in a standard format (as generated by the [OUTPUT](<output.md>) process) from a character external system file.

The format of the file read by **INPDDF** is as follows:

Data Definition

Contents  |  Format
---|---
<file name> |  20A4 (not used)
<file description> |  20A4
<Total fields> |  <Explicit fields> 2I8 (words)

For the total number of fields above, one record per field (or per 4 characters for alphanumeric fields):-

<Field name> |  N/A <length>
---|---
<ISWD pointer> <Char default> |  2A4,1X,A1,2I4,
<numeric default> |  4X,A4,E14.6

The character default is only set for alphanumeric fields; the numeric is only set for numeric fields. The ISWD pointer is absent (zero) for implicit fields, or the sequential word number in the file for explicit fields.

### Data Records

Data records have entries for explicit fields only; all numeric fields will be output in either F or E format, depending on magnitude. Each number will be 12 characters long (e.g. F12.0). All alphanumeric fields will be read in multiples of A4, without spaces, except where an alphanumeric field follows a numeric field, in which case there will be 4 blanks left before the start of the alphanumeric field.

Errors in the DD format will cause the process to be abandoned with an error message; errors in the data format will cause each record in error to be displayed with an error message, and the record ignored.

The maximum record width that may be handled is 240 characters.

The characters !! in positions 1 and 2 are used internally to terminate the data; therefore no record should start with !!.

The following messages are output to the display when using INPDDF with a catalogue file input:

## >>> OPERATING ON A CATALOGUE FILE INPUT <<<

>>> COPYING IN FILE file1
>>> 99999999 RECORDS IN FILE file 1
>>> COPYING IN FILE file2
>>> 99999999 RECORDS IN FILE file 2

### Input Files:

### Output Files:

* **out** (Table):
  Database file to be created. If OUT is a catalogue file, then all files in the
  catalogue will be input.
  Required=Yes

### Fields:

### Parameters:

* **print**:
  >=1 to display each record (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('inpddf')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute inpddf
print("Running inpddf...")
dm_fil.inpddf(
    out_o='t_inpddf_out',  # required
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("inpddf execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_inpddf_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")